# Non-Contractual Player Churn Analytics & Predictive Pipeline

## 1. Business Understanding & Problem Formulation

What do you think the cost of acquiring a new customer for betting firms is? 
A whooping [$3.9 Billion](https://www.5wpr.com/new/the-gambling-industry-spent-3-9-billion-on-marketing-last-year-here-is-what-the-data-shows/) spread affross different advertisement portfolios to get users to walk through their front door. However, they are all faced with one problem, after filling in the last registration form, getting users to continuously place bets has become another matter entirely. [Data](https://igamingbusiness.com/marketing-affiliates/dot-player-churn-a-real-problem/) suggests only 63% of users active for more than 2 years, and only a third of new users make it to the third year still using betting services.   

The aim of this process is to better understand user behavior through rigorous exploratory analysis and model customer behaviour to predict user churning and possibly reduce churning. 

For subscription based services like Netflix, a churn can be defined as a user cancelling their subscription, usually days in advance with a physical action of clicking an unsubscribe button. 
However, this is not the case for betting companies. A user's churn is usually marked by abandoning an account and ceasing to place bets, which does very little in announcing depature. 

In this study, churn is defined as a player's failure to place a bet or make a deposit for a 45-day target period following a fixed observation cutoff date ($T_{\text{cutoff}}$). This 45-day window represents a valid business compromise, accounting for the fact that active players often pause betting due to normal day-to-day life events or seasonal sports breaks. Using $T_{\text{cutoff}}$ as our target boundary, player behavior leading up to departure is analyzed and the factors driving disengagement are modeled.

The formula is as follows

$$Y_i = \begin{cases} 1 & \text{if } \sum_{t = T_{\text{cutoff}}}^{T_{\text{cutoff}} + 45} \left(\text{Bets}_{i,t} + \text{Deposits}_{i,t}\right) = 0 \\ 0 & \text{otherwise} \end{cases}$$

## Data Understanding

In this phase, the size, memory requirements, cardinality, temporal and logical anomalies are identified. 


In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import duckdb

In [2]:
# Connecting to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

In [3]:
# Grouping the datasets into a dictionary for easy access
data = {
    'Demographics': 'datasets/SecondSeshData1Demographics.csv',
    'CashGames': 'datasets/SecondSeshData2CashGames.csv',
    'Tournaments': 'datasets/SecondSeshData3Tournaments.csv',
    'Deposits': 'datasets/SecondSeshData4Deposits.csv',
    'Withdrawals': 'datasets/SecondSeshData5Withdrawals.csv',
}

In [4]:
for table_name, file_path in data.items():
    query = f"CREATE OR REPLACE VIEW {table_name} AS SELECT * FROM read_csv_auto('{file_path}')"
    con.execute(query)

print("All CSV views registered successfully in DuckDB!")

All CSV views registered successfully in DuckDB!


In [5]:
# See all registered views
con.execute("SHOW TABLES").fetchdf()

,name
0,CashGames
1,Demographics
2,Deposits
3,Tournaments
4,Withdrawals


In [6]:
# Check shape,number of users, and min/max dates for each table
audit_query = """
SELECT 
    'Demographics' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT UserID) AS unique_users,
    NULL AS min_date,
    NULL AS max_date
FROM Demographics

UNION ALL

SELECT 
    'CashGames',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(Date AS TIMESTAMP)),
    MAX(CAST(Date AS TIMESTAMP))
FROM CashGames

UNION ALL

SELECT 
    'Tournaments',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(Date AS TIMESTAMP)),
    MAX(CAST(Date AS TIMESTAMP))
FROM Tournaments

UNION ALL

SELECT 
    'Deposits',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(ProcessDate AS TIMESTAMP)),
    MAX(CAST(ProcessDate AS TIMESTAMP))
FROM Deposits

UNION ALL

SELECT 
    'Withdrawals',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(ProcessDate AS TIMESTAMP)),
    MAX(CAST(ProcessDate AS TIMESTAMP))
FROM Withdrawals;
"""

# Run the query and fetch into a Pandas dataframe
audit_results = con.execute(audit_query).fetchdf()
audit_results

,table_name,total_rows,unique_users,min_date,max_date
0,Demographics,5028,5028,NaT,NaT
1,CashGames,51763,4232,2015-02-01,2020-07-28
2,Tournaments,82831,3448,2015-02-01,2021-01-01
3,Deposits,295088,5008,2015-02-01,2020-07-15
4,Withdrawals,32307,1827,2015-02-01,2020-07-15


In [ ]:
# Are there transacting users missing from Demographics? If so, how many?
orphaned_users_query = """
-- Check for transaction users missing from Demographics
SELECT 'Orphan CashGame Users' AS issue_type, COUNT(DISTINCT UserID) AS missing_count
FROM CashGames 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Deposit Users', COUNT(DISTINCT UserID)
FROM Deposits 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Withdrawal Users', COUNT(DISTINCT UserID)
FROM Withdrawals 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Tournament Users', COUNT(DISTINCT UserID)
FROM Tournaments 
WHERE UserID NOT IN (SELECT UserID FROM Demographics);
"""
orphaned_users_results = con.execute(orphaned_users_query).fetchdf()
orphaned_users_results

,issue_type,missing_count
0,Orphan CashGame Users,0
1,Orphan Deposit Users,0
2,Orphan Withdrawal Users,0
3,Orphan Tournament Users,0


,table_name,total_rows,non_null_users,null_user_count
0,Demographics,5028,5028,0
1,CashGames,51763,51763,0
2,Tournaments,82831,82831,0
3,Deposits,295088,295088,0
4,Withdrawals,32307,32307,0


In [24]:
missing_demographics_query = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(UserID) AS missing_user_ids,
    COUNT(*) - COUNT(SystemAgeAsOfReg) AS missing_reg_dates,
    COUNT(*) - COUNT(CountryID) AS missing_countries,
    COUNT(*) - COUNT(Gender) AS missing_genders
FROM Demographics;
"""

missing_demographics_results = con.execute(missing_demographics_query).fetchdf()
missing_demographics_results

,total_rows,missing_user_ids,missing_reg_dates,missing_countries,missing_genders
0,5028,0,0,0,0


In [25]:

missing_transaction_data_query = """
SELECT 
    'CashGames' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(UserID) AS missing_user_ids,
    COUNT(*) - COUNT(Date) AS missing_dates,
    COUNT(*) - COUNT(Windows) AS missing_Windows,
    COUNT(*) - COUNT(StakesC) AS missing_StakesC,
    COUNT(*) - COUNT(WinningsC) AS missing_WinningsC
FROM CashGames
"""
missing_transaction_data = con.execute(missing_transaction_data_query).fetchdf()
missing_transaction_data

,table_name,total_rows,missing_user_ids,missing_dates,missing_Windows,missing_StakesC,missing_WinningsC
0,CashGames,51763,0,0,0,0,0
